# Prototype Connect Four battle

Pit LLMs against each other in a game of Connect Four

In [1]:
from openai import OpenAI
from dotenv import load_dotenv
import json
import os

In [2]:
load_dotenv(override=True)

True

In [3]:
# Some constants

RED = 1
YELLOW = -1
EMPTY = 0
show = {EMPTY:"⚪️", RED: "🔴", YELLOW: "🟡"}
pieces = {EMPTY: "empty", RED: "red", YELLOW: "yellow"}
cols = "ABCDEFG"

In [4]:
# The Game Board
class Board:

    def __init__(self):
        self.cells = [[EMPTY for _ in range(7)] for _ in range(6)]
        self.player = RED
        self.winner = EMPTY

    def __repr__(self):
        result = ""
        for y in range(6):
            for x in range(7):
                result += show[self.cells[5-y][x]]
            result += "\n"
        if self.winner:
            result += f"\n{show[self.winner]} wins\n"
        else:
            result += f"\n{show[self.player]} to play\n"
        return result

    def json(self):
        result = "{\n"
        result += '    "Column names": ["A", "B", "C", "D", "E", "F", "G"],\n'
        for y in range(6):
            result += f'    "Row {6-y}": [' 
            for x in range(7):
                result += f'"{pieces[self.cells[5-y][x]]}", '
            result = result[:-2] + '],\n'
        result = result[:-2]+'\n}'
        return result  

In [5]:
# See the game board

Board()

⚪️⚪️⚪️⚪️⚪️⚪️⚪️
⚪️⚪️⚪️⚪️⚪️⚪️⚪️
⚪️⚪️⚪️⚪️⚪️⚪️⚪️
⚪️⚪️⚪️⚪️⚪️⚪️⚪️
⚪️⚪️⚪️⚪️⚪️⚪️⚪️
⚪️⚪️⚪️⚪️⚪️⚪️⚪️

🔴 to play

In [6]:
# And the json representation

print(Board().json())

{
    "Column names": ["A", "B", "C", "D", "E", "F", "G"],
    "Row 6": ["empty", "empty", "empty", "empty", "empty", "empty", "empty"],
    "Row 5": ["empty", "empty", "empty", "empty", "empty", "empty", "empty"],
    "Row 4": ["empty", "empty", "empty", "empty", "empty", "empty", "empty"],
    "Row 3": ["empty", "empty", "empty", "empty", "empty", "empty", "empty"],
    "Row 2": ["empty", "empty", "empty", "empty", "empty", "empty", "empty"],
    "Row 1": ["empty", "empty", "empty", "empty", "empty", "empty", "empty"]
}


In [7]:
# Some useful methods

def height(self, x):
    height = 0
    while height<6 and self.cells[height][x] != EMPTY:
        height += 1
    return height

def legal_moves(self):
    return [cols[x] for x in range(7) if self.height(x)<6]

def move(self, x):
    self.cells[self.height(x)][x] = self.player
    self.player = -1 * self.player

Board.height = height
Board.legal_moves = legal_moves
Board.move = move

In [8]:
b = Board()
b.move(3)
b.move(3)
b.move(2)
b

⚪️⚪️⚪️⚪️⚪️⚪️⚪️
⚪️⚪️⚪️⚪️⚪️⚪️⚪️
⚪️⚪️⚪️⚪️⚪️⚪️⚪️
⚪️⚪️⚪️⚪️⚪️⚪️⚪️
⚪️⚪️⚪️🟡⚪️⚪️⚪️
⚪️⚪️🔴🔴⚪️⚪️⚪️

🟡 to play

In [9]:
b.legal_moves()

['A', 'B', 'C', 'D', 'E', 'F', 'G']

In [10]:
# Test for winning move

def winning_line(self, x, y, dx, dy):
    color = self.cells[y][x]
    for pointer in range(1, 4):
        xp = x + dx * pointer
        yp = y + dy * pointer
        if not (0 <= xp <= 6 and 0 <= yp <= 5) or self.cells[yp][xp] != color:
            return EMPTY
    return color

def winning_cell(self, x, y):
    for dx, dy in ((0, 1), (1, 1), (1, 0), (1, -1)):
        if winner := self.winning_line(x, y, dx, dy):
            return winner
    return EMPTY

def wins(self):
    for y in range(6):
        for x in range(7):
            if winner := self.winning_cell(x, y):
                return winner
    return EMPTY

def move(self, x):
    self.cells[self.height(x)][x] = self.player
    if winner := self.wins():
        self.winner = winner
    else:
        self.player = -1 * self.player
    return self

Board.winning_line = winning_line
Board.winning_cell = winning_cell
Board.wins = wins
Board.move = move

In [11]:
b = Board()
b.move(2).move(3).move(2).move(3).move(2).move(3).move(2)

⚪️⚪️⚪️⚪️⚪️⚪️⚪️
⚪️⚪️⚪️⚪️⚪️⚪️⚪️
⚪️⚪️🔴⚪️⚪️⚪️⚪️
⚪️⚪️🔴🟡⚪️⚪️⚪️
⚪️⚪️🔴🟡⚪️⚪️⚪️
⚪️⚪️🔴🟡⚪️⚪️⚪️

🔴 wins

In [12]:
class Player:

    def __init__(self, model, color):
        self.color = color
        self.model = model
        openrouter_api_key = os.getenv("OPENROUTER_API_KEY")
        self.llm = OpenAI(
            base_url="https://openrouter.ai/api/v1",
            api_key=openrouter_api_key,
        )

    def system(self, board):
        legal_moves = ", ".join(board.legal_moves())
        return f"""You are an expert player of the board game Connect 4.
                    Players take turns to drop counters into one of 7 columns labelled A, B, C, D, E, F, G.
                    The winner is the first player to get 4 coins in a row in a straight or diagonal line.
                    You are playing with the {pieces[self.color]} coins.
                    And your opponent is playing with the {pieces[self.color * -1]} coins.
                    You will be presented with the board and asked to pick a column to drop your piece.
                    You must pick one of the following legal moves: {legal_moves}. You must pick one of those letters.
                    You should respond in JSON, and only in JSON, according to this spec:

                    {{
                        "evaluation": "brief assessment of the board",
                        "threats": "any threats from your opponent or weaknesses in your position",
                        "opportunities": "any opportunities to gain the upper hand or strengths in your position",
                        "strategy": "the thought process behind your next move",
                        "move_column": "one letter from this list of legal moves: {legal_moves}"
                    }}

                    IMPORTANT: Respond with raw JSON only. Do NOT include markdown code blocks like ```json or ```."""

    def user(self, board):
        legal_moves = ", ".join(board.legal_moves())
        return f"""It is your turn to make a move as {pieces[self.color]}.
                    The current board position is:

                    {board.json()}

                    Now with this in mind, make your decision. Respond only in raw JSON (no markdown code blocks) strictly according to this spec:

                    {{
                        "evaluation": "brief assessment of the board",
                        "threats": "any threats from your opponent or weaknesses in your position",
                        "opportunities": "any opportunities to gain the upper hand or strengths in your position",
                        "strategy": "the thought process behind your next move",
                        "move_column": "one of {legal_moves} which are the legal moves"
                    }}

                    You must pick one of these letters for your move_column: {legal_moves}

                    IMPORTANT: Raw JSON only - no markdown, no ```json or ``` wrapper."""

    def clean_json_response(self, reply):
        """Remove markdown code blocks if present"""
        reply = reply.strip()
        # Remove ```json and ``` wrappers
        if reply.startswith("```json"):
            reply = reply[7:]
        elif reply.startswith("```"):
            reply = reply[3:]
        if reply.endswith("```"):
            reply = reply[:-3]
        return reply.strip()

    def process_move(self, board, reply):
        print(reply)
        try:
            # Clean any markdown before parsing
            clean_reply = self.clean_json_response(reply)
            result = json.loads(clean_reply)
            move = result.get("move_column") or ""
            move = move.upper()
            col = cols.find(move)
            if not (0 <= col <= 6) or board.height(col)==6:
                raise ValueError("Illegal move")
            board.move(col)
        except Exception as e:
            print(f"Exception {e}")
            board.winner = -1 * board.player
            
    
    def move(self, board):
        system = self.system(board)
        user = self.user(board)
        reply = self.llm.chat.completions.create(
            model=self.model,
            messages=[
                {"role": "system", "content": system},
                {"role": "user", "content": user}
            ],
            response_format={"type": "json_object"}
            
        )
        self.process_move(board, reply.choices[0].message.content)

# Let's do this!

Wrap it in a loop, and we're off!

In [ ]:
MODEL_PLAYER1 = "openai/gpt-5.4"
MODEL_PLAYER2 = "anthropic/claude-sonnet-4.5"

board = Board()
red = Player(MODEL_PLAYER1, RED)
yellow = Player(MODEL_PLAYER2, YELLOW)
while not board.winner:
    red.move(board)
    print(board)
    if not board.winner:
        yellow.move(board)
        print(board)

{"evaluation":"The board is completely empty, so the position is balanced and the opening move is important.","threats":"There are no immediate threats from yellow on the first move.","opportunities":"Playing in the center gives the best long-term control and creates the most possible horizontal, vertical, and diagonal connections.","strategy":"Take the center column to maximize future winning lines and maintain strong positional advantage from the start.","move_column":"D"}
⚪️⚪️⚪️⚪️⚪️⚪️⚪️
⚪️⚪️⚪️⚪️⚪️⚪️⚪️
⚪️⚪️⚪️⚪️⚪️⚪️⚪️
⚪️⚪️⚪️⚪️⚪️⚪️⚪️
⚪️⚪️⚪️⚪️⚪️⚪️⚪️
⚪️⚪️⚪️🔴⚪️⚪️⚪️

🟡 to play

{
    "evaluation": "This is the opening position with red having made the first move in column D. The board is nearly empty with plenty of strategic options available.",
    "threats": "No immediate threats as red only has one piece on the board. However, red has claimed the center column which is strategically valuable in Connect 4.",
    "opportunities": "I have the opportunity to contest the center area or build